In [ ]:
from pathlib import Path

from IPython.display import display

from pair_recon.app.cli import prompts
from pair_recon.app.config import (
    load_defaults,
    load_environment,
    save_defaults,
)
from pair_recon.app.runner import launch, write_request_snapshot
from pair_recon.mail import send_outlook
from pair_recon_left.common.runtime.run_logging import configure_console_logging

# Reuse the same environment, prompts, defaults, runner, and logging as run.cmd.
load_environment()
ROOT = next(
    path for path in (Path.cwd(), Path.cwd().parent)
    if (path / "pyproject.toml").is_file()
)
DEFAULTS_FILE = ROOT / ".pair_recon.yaml"
LOG_LEVEL = "INFO"
configure_console_logging(LOG_LEVEL)

In [ ]:
# Run this cell and answer the same interactive questions used by the CLI.
defaults = load_defaults(DEFAULTS_FILE)
settings, config, action, client, catalog = prompts(defaults)
display(
    {
        "settings": {key: value for key, value in settings.items() if not key.startswith("_")},
        "pair": next(iter(config["pairs"])),
        "action": action,
    }
)

In [ ]:
# Set RUN=True only after reviewing the settings shown above.
RUN = False

if not RUN:
    print("Preview only. Set RUN = True to write the request snapshot and launch.")
else:
    snapshot = write_request_snapshot(settings, config)
    save_defaults(
        DEFAULTS_FILE,
        {
            key: value
            for key, value in settings.items()
            if key != "run_id" and not key.startswith("_")
        },
    )
    result = launch(
        settings,
        config,
        action=action,
        workspace_client=client,
        catalog=catalog,
    )
    result["request_snapshot"] = str(snapshot)
    if settings.get("email_to"):
        try:
            result["email"] = send_outlook(
                settings["email_to"],
                f"pair_recon {result['state']}: {result['pair']} {result['run_id']}",
                str(result),
                [result["left_log"]],
            )
        except Exception as exc:
            result["email"] = {"status": "failed", "error": str(exc)}
    display(result)